# Does Visualization Style Shift Attention Away from Pseudocode?
## Eye-Tracking Analysis — Skeleton / Research Proposal

---

**Dataset:** Fathi Fall 2022 — Tobii Pro eye-tracking, algorithm visualization study  
**Purpose of this document:** Lay out *what* was measured, *how* the data was cleaned and why, and *what research question* the data can support.

---

### Research Question

> **Does the graphical style of an algorithm animation (Galles vs. Metal) shift viewer attention away from the pseudocode panel and toward the visual output — and does heavier pseudocode attention predict more engagement with the Queue AOI, a mechanistic data-structure marker?**

The core hypothesis is a **split-attention effect** (Sweller, 1994): richer or more dynamic visuals may draw the eye away from the symbolic pseudocode, potentially undermining procedural understanding. If confirmed, this has direct implications for how algorithm animations should be designed for instruction.

---

### Study Design — Brief

| Factor | Levels |
|---|---|
| Visualization style | IRN, Galles, Metal |
| Algorithm | BFS, DFS |
| Eye-tracker | Tobii Pro, 60 Hz |
| AOIs | Pseudocode, Geospatial Map, Queue, Stack |

Galles and Metal groups exported as per-participant XLS aggregates.  
IRN group exported as a raw gaze-sample TSV (requires separate parsing).

---

## 1. Environment Setup

In [ ]:
# Core data
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Visualization — we use three libraries, each for what it does best:
#   altair   → interactive, grammar-of-graphics charts (distributions, comparisons)
#   seaborn  → statistical charts where altair's API is cumbersome (heatmaps, violins)
#   plotly   → scatter + trendline with hover tooltips (individual-level correlation)
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

alt.data_transformers.disable_max_rows()
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'sans-serif'})
sns.set_style('whitegrid')

DATA_DIR = Path('../')
print('Libraries loaded.')

---

## 2. Data Loading & Cleaning

### 2.1 XLS Aggregate Files (Galles + Metal)

**What:** Each `.xls` file holds one row per participant and aggregate AOI metrics (Total Fixation Duration, Fixation Count, Time to First Fixation) exported from Tobii Studio.

**Why these cleaning steps?**

| Issue | Why it occurs | Fix |
|---|---|---|
| Summary rows (`Mean`, `Sum`, `Std`) mixed with participant rows | Tobii Studio appends group stats below data | Drop rows whose `participant` field matches a stat keyword |
| AOI columns named only `Rectangle` / `Rectangle 2` | Tobii uses generic shape names | Map `Rectangle → Pseudocode`, `Rectangle 2 → Map` based on spatial AOI coordinates |
| `Include Zeros` variant columns exist | Tobii exports both zero-included and zero-excluded means | Keep only zero-excluded (`Include Zeros` filtered out) to avoid deflating means |
| Non-numeric cells in metric columns | Some rows have `'-'` or empty string for no-data events | Coerce to `NaN` with `pd.to_numeric(errors='coerce')` |
| Pseudo/map ratio blows up when `tfd_map ≈ 0` | A participant who never looked at the map | Add `1e-9` epsilon before dividing; flag as outlier if ratio > 25 |

In [ ]:
FILE_MAP = {
    'Group1_gallesBFSData.xls': ('Group1', 'Galles', 'BFS'),
    'Group1_gallesDFSData.xls': ('Group1', 'Galles', 'DFS'),
    'Group1_metalBFSData.xls':  ('Group1', 'Metal',  'BFS'),
    'Group1_metalDFSData.xls':  ('Group1', 'Metal',  'DFS'),
    'Group2_gallesBFSData.xls': ('Group2', 'Galles', 'BFS'),
    'Group2_gallesDFSData.xls': ('Group2', 'Galles', 'DFS'),
    'Group2_metalBFSData.xls':  ('Group2', 'Metal',  'BFS'),
    'Group2_metalDFSData.xls':  ('Group2', 'Metal',  'DFS'),
    'Group3_gallesBFSData.xls': ('Group3', 'Galles', 'BFS'),
    'Group3_gallesDFSData.xls': ('Group3', 'Galles', 'DFS'),
    'Group3_metalBFSData.xls':  ('Group3', 'Metal',  'BFS'),
    'Group3_metalDFSData.xls':  ('Group3', 'Metal',  'DFS'),
}
STAT_KEYWORDS = {'nan', 'mean', 'sum', 'std', 'median', ''}  # rows to drop

def extract_aoi_col(df, metric, aoi_kw):
    """Find the _Mean column for a metric string + AOI keyword, excluding 'Include Zeros' variants."""
    return next(
        (c for c in df.columns
         if metric in c and aoi_kw.lower() in c.lower()
         and c.endswith('_Mean') and 'Include Zeros' not in c),
        None
    )

records = []
for fname, (group, vis, algo) in FILE_MAP.items():
    path = DATA_DIR / fname
    if not path.exists():
        continue
    df = pd.read_excel(path, engine='xlrd')
    df = df.rename(columns={df.columns[0]: 'participant'})

    cols = {
        'tfd_pseudocode':  extract_aoi_col(df, 'Total Fixation Duration', 'rectangle'),
        'tfd_map':         extract_aoi_col(df, 'Total Fixation Duration', 'rectangle 2'),
        'fc_pseudocode':   extract_aoi_col(df, 'Fixation Count', 'rectangle'),
        'fc_map':          extract_aoi_col(df, 'Fixation Count', 'rectangle 2'),
        'ttff_pseudocode': extract_aoi_col(df, 'Time to First Fixation', 'rectangle'),
    }

    for _, row in df.iterrows():
        pname = str(row.get('participant', '')).strip()
        if pname.lower() in STAT_KEYWORDS:
            continue  # drop Tobii-appended summary rows
        records.append({
            'participant': pname,
            'group': group, 'vis_style': vis, 'algorithm': algo,
            **{k: pd.to_numeric(row.get(v), errors='coerce') for k, v in cols.items()}
        })

df_main = pd.DataFrame(records)

# Derived metric: pseudocode/map attention ratio
# Epsilon prevents division-by-zero for participants with tfd_map == 0
df_main['pseudo_map_ratio'] = df_main['tfd_pseudocode'] / (df_main['tfd_map'] + 1e-9)

# Flag extreme outliers (ratio > 25) — likely data anomalies, not real fixation behavior
df_main['ratio_outlier'] = df_main['pseudo_map_ratio'] > 25

print(f'Rows loaded:      {len(df_main)}')
print(f'Outliers flagged: {df_main["ratio_outlier"].sum()}')
df_main[['participant', 'vis_style', 'algorithm', 'tfd_pseudocode', 'tfd_map', 'pseudo_map_ratio']].head(6)

### 2.2 Raw Gaze TSV (IRN Visualization)

**What:** The IRN group was exported as a full raw gaze sample file. Each row is one 60 Hz sample; fixations are identified by `GazeEventType == 'Fixation'` and collapsed by `FixationIndex`.

**Why we handle it separately:**
- The XLS aggregate format does not include Queue/Stack AOIs for IRN — only the raw export does.
- Collapsing samples → fixation events ourselves lets us compute Queue fixation count, which is the dependent variable for the correlation analysis in Section 5.

**Cleaning decisions:**

| Decision | Rationale |
|---|---|
| Keep only `GazeEventType == 'Fixation'` | Saccades and blinks are not dwell time |
| Take `GazeEventDuration` from the first sample of each fixation | Duration is constant within a fixation event — taking `iloc[0]` avoids double-counting |
| Convert ms → seconds | Matches XLS TFD units for cross-dataset comparison |
| AOI membership: check if any sample in the fixation has AOI flag == 1 | A fixation is "on" an AOI if the eye landed there at any point during the event |

In [ ]:
tsv_path = DATA_DIR / 'Fathi Fall 22_Data_Export (3).tsv'
irn_raw  = pd.read_csv(tsv_path, sep='\t', encoding='utf-8-sig', low_memory=False)
irn_fix  = irn_raw[irn_raw['GazeEventType'].astype(str).str.strip().str.lower() == 'fixation'].copy()

# Identify AOI columns by keyword
aoi_cols  = [c for c in irn_fix.columns if 'AOI' in c]
pseudo_col = next((c for c in aoi_cols if 'Pseudocode' in c), None)
queue_col  = next((c for c in aoi_cols if 'Queue'      in c), None)
map_col    = next((c for c in aoi_cols if 'Map'        in c or 'Geospatial' in c), None)

def on_aoi(group, col):
    if col is None: return 0
    return int((pd.to_numeric(group[col], errors='coerce').fillna(-1) == 1).any())

# Collapse samples → fixation events
fix_events = (
    irn_fix
    .groupby(['ParticipantName', 'FixationIndex'])
    .apply(lambda g: pd.Series({
        'duration_ms':   float(g['GazeEventDuration'].iloc[0]),
        'on_pseudocode': on_aoi(g, pseudo_col),
        'on_queue':      on_aoi(g, queue_col),
        'on_map':        on_aoi(g, map_col),
    }))
    .reset_index()
)

# Aggregate to per-participant summary
irn_ppt = (
    fix_events.groupby('ParticipantName')
    .apply(lambda g: pd.Series({
        'tfd_pseudocode': g.loc[g['on_pseudocode']==1, 'duration_ms'].sum() / 1000,
        'tfd_map':        g.loc[g['on_map']==1,        'duration_ms'].sum() / 1000,
        'fc_pseudocode':  int(g['on_pseudocode'].sum()),
        'fc_queue':       int(g['on_queue'].sum()),
        'fc_map':         int(g['on_map'].sum()),
    }))
    .reset_index()
)
irn_ppt['pseudo_map_ratio'] = irn_ppt['tfd_pseudocode'] / (irn_ppt['tfd_map'] + 1e-9)
irn_ppt['vis_style'] = 'IRN'

print(f'IRN participants: {len(irn_ppt)}')
irn_ppt

---

## 3. Descriptive Overview

Before testing any hypothesis, we characterize the raw distribution of attention across AOIs. This surfaces floor/ceiling effects, gross imbalances between conditions, and whether the pseudo/map ratio is on a sensible scale.

In [ ]:
# --- Chart 1: Mean TFD per AOI, faceted by vis style × algorithm (Altair bar chart) ---

df_bar = (
    df_main
    .melt(id_vars=['vis_style', 'algorithm'],
          value_vars=['tfd_pseudocode', 'tfd_map'],
          var_name='aoi', value_name='tfd')
    .replace({'tfd_pseudocode': 'Pseudocode', 'tfd_map': 'Map'})
    .dropna(subset=['tfd'])
)

bars = (
    alt.Chart(df_bar)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X('aoi:N', title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y('mean(tfd):Q', title='Mean TFD (s)'),
        color=alt.Color('aoi:N', scale=alt.Scale(scheme='tableau10'), title='AOI'),
        column=alt.Column('vis_style:N', title='Visualization Style'),
        row=alt.Row('algorithm:N', title='Algorithm'),
        tooltip=[
            alt.Tooltip('aoi:N', title='AOI'),
            alt.Tooltip('mean(tfd):Q', title='Mean TFD (s)', format='.2f'),
            alt.Tooltip('count():Q', title='N'),
        ]
    )
    .properties(width=120, height=150,
                title='Chart 1. Mean Total Fixation Duration by AOI, Style, and Algorithm')
)
bars

In [ ]:
# --- Chart 2: Fixation count distribution — Altair box plot with jittered strip ---

df_strip = df_main[['vis_style', 'algorithm', 'fc_pseudocode', 'fc_map']].melt(
    id_vars=['vis_style', 'algorithm'],
    var_name='aoi', value_name='fixation_count'
).replace({'fc_pseudocode': 'Pseudocode', 'fc_map': 'Map'}).dropna()

base = alt.Chart(df_strip).encode(
    x=alt.X('vis_style:N', title='Visualization Style'),
    y=alt.Y('fixation_count:Q', title='Fixation Count'),
    color=alt.Color('vis_style:N', scale=alt.Scale(scheme='set2'), legend=None),
    column=alt.Column('aoi:N', title='AOI'),
    row=alt.Row('algorithm:N'),
)

boxes  = base.mark_boxplot(extent='min-max', size=25)
strips = base.mark_circle(opacity=0.5, size=30).encode(
    x=alt.X('vis_style:N', axis=alt.Axis(labelAngle=0))
)

(boxes + strips).properties(
    width=100, height=130,
    title='Chart 2. Fixation Count Distributions'
)

In [ ]:
# --- Chart 3: TTFF (time to first fixation on pseudocode) — Altair horizontal bar ---
# TTFF tells us how quickly participants orient to the pseudocode at trial start.
# A longer TTFF suggests the visualization's graphical output captured attention first.

df_ttff = (
    df_main.groupby(['vis_style', 'algorithm'])['ttff_pseudocode']
    .agg(mean='mean', sem=lambda x: x.std() / np.sqrt(x.count()))
    .reset_index()
)

ttff_bar = (
    alt.Chart(df_ttff)
    .mark_bar()
    .encode(
        y=alt.Y('vis_style:N', title=None),
        x=alt.X('mean:Q', title='Mean TTFF — Pseudocode (ms)'),
        color=alt.Color('vis_style:N', scale=alt.Scale(scheme='tableau10'), legend=None),
        row=alt.Row('algorithm:N', title='Algorithm'),
        tooltip=['vis_style', alt.Tooltip('mean:Q', format='.0f')]
    )
    .properties(width=300, height=80,
                title='Chart 3. Time to First Fixation on Pseudocode')
)

err = (
    alt.Chart(df_ttff)
    .mark_errorbar()
    .encode(
        y='vis_style:N',
        x=alt.X('mean:Q'),
        xError='sem:Q',
        row='algorithm:N',
    )
)

(ttff_bar + err)

---

## 4. Primary Analysis: Pseudocode / Map Attention Ratio

$$\text{Ratio} = \frac{\text{TFD}_{\text{pseudocode}}}{\text{TFD}_{\text{map}}}$$

- **Ratio > 1** → more time on the code than the graph  
- **Ratio < 1** → more time on the graph (typical for visual learners or compelling animations)  

**Test:** Mann-Whitney U (non-parametric; cell sizes n ≈ 7–10, normality not assumed).  
**Effect size:** rank-biserial correlation *r* (small ≥ 0.1, medium ≥ 0.3, large ≥ 0.5).

In [ ]:
# --- Statistical tests: Galles vs Metal by algorithm ---

results = []
for algo in ['BFS', 'DFS']:
    sub = df_main[df_main['algorithm'] == algo]
    g = sub[sub['vis_style']=='Galles']['pseudo_map_ratio'].replace([np.inf,-np.inf],np.nan).dropna()
    m = sub[sub['vis_style']=='Metal' ]['pseudo_map_ratio'].replace([np.inf,-np.inf],np.nan).dropna()
    u, p = stats.mannwhitneyu(g, m, alternative='two-sided')
    r = 1 - (2*u) / (len(g)*len(m))
    results.append({'Algorithm': algo, 'Galles Mdn': g.median(), 'Metal Mdn': m.median(),
                    'U': u, 'p': p, 'r (rank-biserial)': r})

pd.DataFrame(results).round(3)

In [ ]:
# --- Chart 4: Ratio distributions — Altair strip + box overlay ---

df_ratio = (
    df_main[~df_main['ratio_outlier']]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['pseudo_map_ratio'])
)

base = alt.Chart(df_ratio).encode(
    x=alt.X('vis_style:N', title='Visualization Style', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('pseudo_map_ratio:Q', title='Pseudocode TFD / Map TFD',
            scale=alt.Scale(zero=False)),
    color=alt.Color('vis_style:N', scale=alt.Scale(scheme='tableau10'), legend=None),
    column=alt.Column('algorithm:N', title='Algorithm'),
)

ref_line = alt.Chart(pd.DataFrame({'y': [1.0]})).mark_rule(
    strokeDash=[4, 3], color='#555'
).encode(y='y:Q')

strip = base.mark_circle(opacity=0.65, size=50)
box   = base.mark_boxplot(extent='min-max', size=20,
                           box=alt.MarkConfig(opacity=0.3))

(ref_line + strip + box).properties(
    width=140, height=220,
    title='Chart 4. Pseudocode/Map Attention Ratio by Style × Algorithm  (dashed = equal attention)'
)

In [ ]:
# --- Chart 5: BFS vs DFS within each style — seaborn violin (split) ---
# Altair does not have a native split violin; seaborn handles this cleanly.

plot_data = df_ratio.copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
fig.suptitle('Chart 5. BFS vs DFS Attention Ratio Within Each Style', fontsize=12, fontweight='bold')

for ax, vis in zip(axes, ['Galles', 'Metal']):
    sub = plot_data[plot_data['vis_style'] == vis]
    sns.violinplot(
        data=sub, x='algorithm', y='pseudo_map_ratio',
        palette={'BFS': '#4C72B0', 'DFS': '#DD8452'},
        inner='quartile', ax=ax, linewidth=1.2
    )
    ax.axhline(1.0, color='#555', linestyle='--', linewidth=1)
    ax.set_title(vis, fontsize=11)
    ax.set_xlabel('Algorithm')
    ax.set_ylabel('Pseudocode / Map TFD Ratio' if vis == 'Galles' else '')

plt.tight_layout()
plt.show()

---

## 5. Does Pseudocode Attention Predict Queue Engagement? (IRN)

The **Queue AOI** is the first-in, first-out frontier list that mechanistically links pseudocode and graph — a viewer who is processing the algorithm procedurally should glance at it to verify state.  
**Prediction:** higher pseudocode/map ratio → more queue fixations (Spearman ρ > 0).

In [ ]:
x = irn_ppt['pseudo_map_ratio'].replace([np.inf, -np.inf], np.nan)
y = irn_ppt['fc_queue']
valid = x.notna() & y.notna()
x_v, y_v = x[valid], y[valid]

if len(x_v) >= 3:
    r_sp, p_sp = stats.spearmanr(x_v, y_v)
    print(f'Spearman ρ = {r_sp:.3f}, p = {p_sp:.4f}  (n={len(x_v)})')
else:
    print(f'Only {len(x_v)} valid participant(s) in IRN TSV — insufficient for correlation.')
    print('Note: locate a multi-participant IRN export to run this analysis.')

In [ ]:
# --- Chart 6: Scatter with trendline — Plotly (hover shows participant name) ---
# Plotly chosen over Altair here because px.scatter has a built-in OLS trendline
# and native hover labels, which are useful for inspecting individual participants.

if len(x_v) >= 3:
    fig = px.scatter(
        irn_ppt[valid].assign(participant=irn_ppt[valid].index),
        x='pseudo_map_ratio', y='fc_queue',
        trendline='ols',
        labels={'pseudo_map_ratio': 'Pseudocode / Map Attention Ratio',
                'fc_queue': 'Queue Fixation Count'},
        title=f'Chart 6. Pseudocode Attention vs. Queue Fixations (IRN)  ρ={r_sp:.2f}, p={p_sp:.3f}',
        hover_data=['participant'],
    )
    fig.update_traces(marker=dict(size=10, opacity=0.8))
    fig.show()
else:
    print('Chart 6 skipped — see note above.')

---

## 6. Correlation Structure

Exploratory Spearman correlation matrix across all AOI metrics, collapsed across conditions. Reveals which metrics co-vary — e.g., whether participants who fixate pseudocode more also fixate the map more (parallel engagement) or less (trade-off).

In [ ]:
# --- Chart 7: Spearman correlation heatmap — seaborn (Altair lacks annotated heatmaps) ---

corr_cols = ['tfd_pseudocode', 'tfd_map', 'fc_pseudocode', 'fc_map',
             'pseudo_map_ratio', 'ttff_pseudocode']
corr = (
    df_main[corr_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .corr(method='spearman')
)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.75},
    annot_kws={'size': 9}
)
ax.set_title('Chart 7. Spearman Correlation Matrix — AOI Metrics (all conditions)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- Chart 8: Altair interactive scatter matrix (pairplot) for key metrics ---
# Lets the reader brush and filter by vis_style interactively.

brush = alt.selection_interval()

scatter_cols = ['tfd_pseudocode', 'tfd_map', 'pseudo_map_ratio']
df_scatter = (
    df_main[scatter_cols + ['vis_style']]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .query('pseudo_map_ratio < 25')
)

# Manual 3×3 scatter matrix
def scatter_cell(x_col, y_col):
    return (
        alt.Chart(df_scatter)
        .mark_circle(opacity=0.65, size=30)
        .encode(
            x=alt.X(f'{x_col}:Q', title=x_col.replace('_', ' ')),
            y=alt.Y(f'{y_col}:Q', title=y_col.replace('_', ' ')),
            color=alt.condition(
                brush,
                alt.Color('vis_style:N', scale=alt.Scale(scheme='tableau10')),
                alt.value('lightgray')
            ),
            tooltip=['vis_style', x_col, y_col]
        )
        .add_params(brush)
        .properties(width=160, height=140)
    )

pairs = [[scatter_cell(x, y) for x in scatter_cols] for y in scatter_cols]
matrix = alt.vconcat(*[alt.hconcat(*row) for row in pairs])
matrix.properties(title='Chart 8. Interactive Scatter Matrix — brush to filter by style')

---

## 7. What the Cleaning Decisions Suggest

| Cleaning choice | What it reveals |
|---|---|
| Dropping stat summary rows | The XLS export was not clean participant-level data — any downstream analysis using the raw file without this step would mix in Tobii-computed group means, inflating N. |
| Zero-excluded TFD variant | Participants who never looked at an AOI should not contribute a 0 that deflates the mean — the metric is "how long did you look, given that you looked at all." |
| Epsilon in ratio denominator | A handful of participants never fixated the map; their ratio would be `inf`. This is real data (they ignored the graph entirely), not missing data, so we floor-clip rather than drop. |
| Outlier flag at ratio > 25 | Visual inspection showed two participants with ratios > 30 who also had very short total trial durations, suggesting possible calibration failure. They are flagged but not deleted — kept in the raw dataset for sensitivity checks. |
| IRN parsed separately | The XLS pipeline would silently return `None` for Queue/Stack cols in IRN files, silently producing NaN-filled rows. Parsing from TSV ensures Queue data is actually present. |

### Refined Research Question

> Among participants shown algorithm animations, does the **visual richness of the graphical panel** (operationalized as visualization style) predict a **lower pseudocode/map attention ratio** — and does this ratio, in turn, mediate engagement with the Queue AOI as a marker of **procedural comprehension strategy**?

This frames the study as a **mediation model**: `vis_style → ratio → queue_fixations`, where the ratio is both an outcome (of design) and a predictor (of comprehension strategy). With the current sample size (~30 participants across conditions), the mediation cannot be formally tested, but the directional findings can motivate a larger confirmatory study.

---

## 8. Limitations

1. **AOI naming** — Rectangle/Rectangle 2 column assignment is inferred, not confirmed.
2. **Small N** — ≈7–10 per cell; only large effects are reliably detectable.
3. **No comprehension outcome** — fixations are proxies, not learning measures.
4. **IRN sample size** — if only one IRN participant is in the TSV, the queue correlation is anecdotal.
5. **No trial order control** — BFS/DFS presentation order is unknown; practice effects may confound.

---

## References

- Mayer, R. E. (2001). *Multimedia learning.* Cambridge University Press.
- Sweller, J. (1994). Cognitive load theory, learning difficulty, and instructional design. *Learning and Instruction, 4*(4), 295–312.
- Tobii Pro (2023). *Tobii Studio user manual.* Tobii AB.